In [9]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [10]:

# Import required libraries
import ollama
import time


# Select local LLM model
MODEL_NAME = "llama3.2"


In [11]:
def ensure_model_available(model_name: str):
    """Check that the model exists locally; pull it if it doesn't."""
    local_models = [m["name"] if "name" in m else m["model"] for m in ollama.list().get("models", [])]
    if not any(model_name in m for m in local_models):
        print(f"Model '{model_name}' not found locally. Pulling it now...")
        ollama.pull(model_name)
        print("Pull complete.")
    else:
        print(f"Model '{model_name}' is available locally.")

ensure_model_available(MODEL_NAME)

Model 'llama3.2' is available locally.


In [12]:
def generate_response(prompt: str, model: str = MODEL_NAME, temperature: float = 0.7):
    """Send a prompt to the local LLM and return the response text + stats."""
    start = time.time()
    result = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": temperature},
    )
    elapsed = time.time() - start

    reply_text = result["message"]["content"]
    stats = {
        "model": model,
        "temperature": temperature,
        "time_seconds": round(elapsed, 2),
        "prompt_tokens": result.get("prompt_eval_count"),
        "response_tokens": result.get("eval_count"),
    }
    return reply_text, stats

In [13]:
user_prompt = input("Enter your prompt: ")

reply, stats = generate_response(user_prompt)

print("\n--- PROMPT ---")
print(user_prompt)
print("\n--- RESPONSE ---")
print(reply)
print("\n--- STATS ---")
print(stats)

Enter your prompt:  hi



--- PROMPT ---
hi

--- RESPONSE ---
How can I assist you today?

--- STATS ---
{'model': 'llama3.2', 'temperature': 0.7, 'time_seconds': 5.59, 'prompt_tokens': 26, 'response_tokens': 8}


In [ ]:
test_prompts = [
    "Explain Artificial Intelligence in simple words.",
    "Write a four-line poem about technology.",
    "If a student studies for 3 hours every day for 7 days, how many hours did the student study? Explain your calculation."
]
results = []
for p in test_prompts:
    reply, stats = generate_response(p)
    results.append({"prompt": p, "response": reply, "stats": stats})
    print(f"PROMPT: {p}")
    print(f"RESPONSE: {reply}")
    print(f"STATS: {stats}")
    print("-" * 80)

In [ ]:
# Compare the same prompt at different temperatures (low = focused/deterministic, high = more random/creative)
temperature_prompt = "Describe a rainy day in one sentence."

for temp in [0.0, 0.9]:
    reply, stats = generate_response(temperature_prompt, temperature=temp)
    print(f"Temperature = {temp}")
    print(f"RESPONSE: {reply}")
    print(f"STATS: {stats}")
    print("-" * 80)

In [ ]:
import requests

def generate_response_rest(prompt: str, model: str = MODEL_NAME):
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
    )
    data = response.json()
    return data["message"]["content"]

# Example:
# print(generate_response_rest("Explain what an LLM is in one sentence."))